# 듣다 — OpenL3 Audio Embedding PoC (Colab)
embedding_pending.csv 업로드 → OpenL3 임베딩 → generated_embeddings.json 다운로드.
GPU 불필요(CPU 런타임). service_role key 불필요(임포트는 관리자 앱에서 수행).

In [ ]:
# 1) 의존성 설치 (런타임당 1회)
!pip -q install openl3 soundfile requests numpy

In [ ]:
# 2) embedding_pending.csv 업로드
from google.colab import files
import csv, io
up = files.upload()  # embedding_pending.csv 선택
name = next(iter(up))
rows = list(csv.DictReader(io.StringIO(up[name].decode('utf-8'))))
print(f'{len(rows)} tracks loaded')

In [ ]:
# 3) OpenL3 임베딩 생성 (512d, 전체 평균풀링 + L2 정규화)
import requests, tempfile, numpy as np, soundfile as sf, openl3, json, traceback

MODEL_NAME='openl3'; MODEL_VERSION='v1'; EMB_DIM=512
out=[]; errors=[]
for i, r in enumerate(rows):
    tid=r['track_id']; url=r['audio_url']
    try:
        resp=requests.get(url, timeout=60); resp.raise_for_status()
        with tempfile.NamedTemporaryFile(suffix='.mp3', delete=True) as f:
            f.write(resp.content); f.flush()
            audio, sr = sf.read(f.name)
        if audio.ndim>1: audio=audio.mean(axis=1)  # mono
        emb,_=openl3.get_audio_embedding(audio, sr, content_type='music', embedding_size=EMB_DIM, hop_size=1.0)
        v=emb.mean(axis=0); n=np.linalg.norm(v); v=(v/n if n>0 else v)
        out.append({'track_id':tid,'model_name':MODEL_NAME,'model_version':MODEL_VERSION,'embedding_dim':int(v.shape[0]),'embedding':[round(float(x),6) for x in v.tolist()]})
        print(f'[{i+1}/{len(rows)}] ok {tid}')
    except Exception as e:
        errors.append({'track_id':tid,'reason':str(e)})
        print(f'[{i+1}/{len(rows)}] FAIL {tid}: {e}')
print(f'done: ok={len(out)} fail={len(errors)}')

In [ ]:
# 4) generated_embeddings.json 다운로드
with open('generated_embeddings.json','w') as f:
    json.dump(out, f)
files.download('generated_embeddings.json')
print('관리자 앱 > AI 큐레이션 > 임베딩(PoC) 에서 dry-run 검증 후 임포트하세요.')